In [1]:
import dukascopy_python
from dukascopy_python import instruments as ins

matches = [x for x in dir(ins) if "XAU" in x.upper() or "GOLD" in x.upper()]
print(matches)

['INSTRUMENT_FX_METALS_XAU_USD']


In [2]:
from datetime import datetime

INSTR_NAME = [x for x in matches if "USD" in x][0]
INSTR = getattr(ins, INSTR_NAME)
print("Instrument utilisé :", INSTR_NAME)

df = dukascopy_python.fetch(
    INSTR,
    dukascopy_python.INTERVAL_HOUR_1,
    dukascopy_python.OFFER_SIDE_BID,
    datetime(2025, 1, 1),
    datetime(2025, 2, 1),
)

print("Lignes :", df.shape[0])
print("Période :", df.index.min(), "→", df.index.max(), "| tz :", df.index.tz)
print(df.dtypes)
df.head()

Instrument utilisé : INSTRUMENT_FX_METALS_XAU_USD
Lignes : 504
Période : 2025-01-01 23:00:00+00:00 → 2025-01-31 21:00:00+00:00 | tz : UTC
open      float64
high      float64
low       float64
close     float64
volume    float64
dtype: object


,open,high,low,close,volume
timestamp,,,,,
2025-01-01 23:00:00+00:00,2625.098,2626.005,2621.475,2623.555,0.76854
2025-01-02 00:00:00+00:00,2623.655,2627.765,2622.448,2625.185,1.25358
2025-01-02 01:00:00+00:00,2625.185,2634.148,2623.645,2632.735,3.60279
2025-01-02 02:00:00+00:00,2632.735,2636.298,2632.148,2634.498,2.38464
2025-01-02 03:00:00+00:00,2634.404,2635.198,2632.065,2633.448,1.59480


In [3]:
import time
from datetime import datetime
import pandas as pd

START = datetime(2015, 1, 1)
END   = datetime(2026, 7, 1)   # borne fixe → snapshot reproductible

frames = []
for year in range(START.year, END.year + 1):
    cs = max(datetime(year, 1, 1), START)
    ce = min(datetime(year + 1, 1, 1), END)
    if cs >= ce:
        continue
    t0 = time.time()
    part = dukascopy_python.fetch(
        INSTR,
        dukascopy_python.INTERVAL_HOUR_1,
        dukascopy_python.OFFER_SIDE_BID,
        cs, ce,
    )
    frames.append(part)
    print(f"{year}: {len(part):>6} barres  ({cs.date()} → {ce.date()})  {time.time()-t0:.1f}s")

raw = pd.concat(frames)
print("\nTOTAL brut:", len(raw), "barres")

2015:   5991 barres  (2015-01-01 → 2016-01-01)  1.8s
2016:   5924 barres  (2016-01-01 → 2017-01-01)  1.1s
2017:   5910 barres  (2017-01-01 → 2018-01-01)  0.7s
2018:   5906 barres  (2018-01-01 → 2019-01-01)  1.0s
2019:   5910 barres  (2019-01-01 → 2020-01-01)  1.5s
2020:   5927 barres  (2020-01-01 → 2021-01-01)  1.3s
2021:   5907 barres  (2021-01-01 → 2022-01-01)  1.2s
2022:   5915 barres  (2022-01-01 → 2023-01-01)  1.2s
2023:   5890 barres  (2023-01-01 → 2024-01-01)  1.4s
2024:   5937 barres  (2024-01-01 → 2025-01-01)  0.6s
2025:   5912 barres  (2025-01-01 → 2026-01-01)  0.6s
2026:   2912 barres  (2026-01-01 → 2026-07-01)  0.4s

TOTAL brut: 68041 barres


In [4]:
import pandas as pd

df = raw.copy()
df = df[~df.index.duplicated(keep="first")]
df = df.sort_index()

print("=== RAPPORT QUALITÉ ===")
print("Barres          :", len(df))
print("Période         :", df.index.min(), "→", df.index.max())
print("Fuseau          :", df.index.tz)
print("Doublons retirés:", len(raw) - len(df))
print("Index croissant :", df.index.is_monotonic_increasing)
print("NaN total       :", int(df.isna().sum().sum()))

bad = (
    (df["high"] < df[["open", "close"]].max(axis=1)) |
    (df["low"]  > df[["open", "close"]].min(axis=1)) |
    (df["high"] < df["low"])
)
print("Violations OHLC :", int(bad.sum()))

gaps = df.index.to_series().diff()
big = gaps[gaps > pd.Timedelta(hours=1)]
wk = big.index.weekday
weekend_like = big[(wk == 6) | (wk == 0)]
print("\nTrous > 1h      :", len(big), "(normal : pause quotidienne + week-ends)")
print("  ~ week-end     :", len(weekend_like))
print("  en semaine     :", len(big) - len(weekend_like))
print("\n10 plus grands trous (reprise → durée) :")
print(big.sort_values(ascending=False).head(10))

print("\nPrix close min/max :", round(df["close"].min(), 2), "/", round(df["close"].max(), 2))

=== RAPPORT QUALITÉ ===
Barres          : 68041
Période         : 2015-01-01 23:00:00+00:00 → 2026-06-30 23:00:00+00:00
Fuseau          : UTC
Doublons retirés: 0
Index croissant : True
NaN total       : 0
Violations OHLC : 0

Trous > 1h      : 2853 (normal : pause quotidienne + week-ends)
  ~ week-end     : 1169
  en semaine     : 1684

10 plus grands trous (reprise → durée) :
timestamp
2015-12-27 23:00:00+00:00   3 days 05:00:00
2020-12-27 23:00:00+00:00   3 days 05:00:00
2017-12-25 23:00:00+00:00   3 days 02:00:00
2022-04-17 22:00:00+00:00   3 days 02:00:00
2019-04-21 22:00:00+00:00   3 days 02:00:00
2017-01-02 23:00:00+00:00   3 days 02:00:00
2018-04-01 22:00:00+00:00   3 days 02:00:00
2025-04-20 22:00:00+00:00   3 days 02:00:00
2022-12-26 23:00:00+00:00   3 days 02:00:00
2023-04-09 22:00:00+00:00   3 days 02:00:00
Name: timestamp, dtype: timedelta64[ns]

Prix close min/max : 1050.01 / 5562.3


In [5]:
import hashlib, json
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version as _pkgver

# Racine du projet (le notebook est dans notebooks/)
PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
SNAP_DIR = PROJECT / "data" / "snapshots"
SNAP_DIR.mkdir(parents=True, exist_ok=True)

VERSION = "v1"
base = f"XAUUSD_H1_BID_2015-01-01_2026-06-30_{VERSION}"
csv_path = SNAP_DIR / f"{base}.csv"
manifest_path = SNAP_DIR / f"{base}.manifest.json"

# Contrôles recalculés → manifeste auto-suffisant et honnête
bad = (
    (df["high"] < df[["open", "close"]].max(axis=1)) |
    (df["low"]  > df[["open", "close"]].min(axis=1)) |
    (df["high"] < df["low"])
)
gaps = df.index.to_series().diff()
big = gaps[gaps > pd.Timedelta(hours=1)]

df.to_csv(csv_path)  # index = timestamp UTC en ISO
sha = hashlib.sha256(csv_path.read_bytes()).hexdigest()

manifest = {
    "instrument": "XAUUSD — or spot",
    "instrument_constant": INSTR_NAME,
    "source": "Dukascopy Bank SA (via dukascopy-python)",
    "library_version": _pkgver("dukascopy-python"),
    "offer_side": "BID",
    "interval": "H1",
    "timezone": "UTC",
    "start": str(df.index.min()),
    "end": str(df.index.max()),
    "n_bars": int(len(df)),
    "columns": list(df.columns),
    "price_close_min": round(float(df["close"].min()), 3),
    "price_close_max": round(float(df["close"].max()), 3),
    "nan_count": int(df.isna().sum().sum()),
    "ohlc_violations": int(bad.sum()),
    "gaps_over_1h": int(len(big)),
    "largest_gap": str(big.max()),
    "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
    "csv_file": csv_path.name,
    "sha256": sha,
    "version": VERSION,
    "note": "IMMUABLE — ne jamais réécrire. Toute nouvelle donnée = nouvelle version.",
}
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

# Vérification round-trip : le fichier gelé se relit sans perte
check = pd.read_csv(csv_path, index_col=0, parse_dates=True)
assert len(check) == len(df) and hashlib.sha256(csv_path.read_bytes()).hexdigest() == sha

print("Snapshot gelé et relu sans perte :", len(check), "barres")
print("Fichier :", csv_path.name)
print("sha256  :", sha)
print("\n" + json.dumps(manifest, indent=2, ensure_ascii=False))

Snapshot gelé et relu sans perte : 68041 barres
Fichier : XAUUSD_H1_BID_2015-01-01_2026-06-30_v1.csv
sha256  : 1c9f7d0349f78879c7ae116febc797a0b225e69c77acdd9f2faac06a8df13606

{
  "instrument": "XAUUSD — or spot",
  "instrument_constant": "INSTRUMENT_FX_METALS_XAU_USD",
  "source": "Dukascopy Bank SA (via dukascopy-python)",
  "library_version": "4.0.1",
  "offer_side": "BID",
  "interval": "H1",
  "timezone": "UTC",
  "start": "2015-01-01 23:00:00+00:00",
  "end": "2026-06-30 23:00:00+00:00",
  "n_bars": 68041,
  "columns": [
    "open",
    "high",
    "low",
    "close",
    "volume"
  ],
  "price_close_min": 1050.008,
  "price_close_max": 5562.295,
  "nan_count": 0,
  "ohlc_violations": 0,
  "gaps_over_1h": 2853,
  "largest_gap": "3 days 05:00:00",
  "downloaded_at_utc": "2026-07-04T21:17:06.122351+00:00",
  "csv_file": "XAUUSD_H1_BID_2015-01-01_2026-06-30_v1.csv",
  "sha256": "1c9f7d0349f78879c7ae116febc797a0b225e69c77acdd9f2faac06a8df13606",
  "version": "v1",
  "note": "IMMUABL